# Módulo 1 – Análisis Exploratorio de Datos (EDA)
## Predicción de Demanda de Transporte Turístico
### IRNA – Universidad Nacional de Colombia

Este notebook realiza el análisis exploratorio completo del dataset de viajes turísticos en Colombia,
identificando patrones temporales, rutas populares y características estadísticas relevantes
para el posterior modelado con LSTM.

In [ ]:
# ─── Celda 1: Importaciones y configuración global ───────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # backend sin display para entornos sin GUI
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')

# Rutas de trabajo
BASE_DIR = Path('.')   # modulo1_demanda/
DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Estilo de graficas
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 13, 'axes.labelsize': 11})
sns.set_theme(style='whitegrid', palette='muted')

print('Librerias cargadas correctamente.')
print(f'Directorio de datos: {DATA_DIR.resolve()}')

In [ ]:
# ─── Celda 2: Carga del dataset (kagglehub o datos sinteticos) ───────────────
CSV_PATH = DATA_DIR / 'travel_data.csv'

def generate_synthetic_data(n=15000, save_path=CSV_PATH):
    """Genera dataset sintetico de viajes turisticos con estacionalidad."""
    np.random.seed(42)
    destinations = [
        'Cartagena', 'Bogota', 'Medellin', 'Santa Marta', 'San Andres',
        'Bucaramanga', 'Cali', 'Barranquilla', 'Pereira', 'Manizales'
    ] + [f'Dest_{i}' for i in range(40)]
    categories   = ['Playa', 'Ciudad', 'Montana', 'Ecoturismo', 'Cultural']
    travel_types = ['Solo', 'Pareja', 'Familia', 'Grupo']
    seasons      = ['Alta', 'Baja', 'Media']

    dates = pd.date_range('2020-01-01', '2024-12-31')
    rows = {
        'user_id':       np.random.randint(1, 501, n),
        'destination':   np.random.choice(destinations, n),
        'category':      np.random.choice(categories, n),
        'country':       'Colombia',
        'rating':        np.random.uniform(2.5, 5.0, n).round(1),
        'visit_date':    np.random.choice(dates, n),
        'num_reviews':   np.random.randint(1, 500, n),
        'avg_cost_usd':  np.random.randint(50, 800, n),
        'travel_type':   np.random.choice(travel_types, n),
        'season':        np.random.choice(seasons, n),
        'duration_days': np.random.randint(1, 15, n),
    }
    df = pd.DataFrame(rows)
    df['visit_date'] = pd.to_datetime(df['visit_date'])

    # Estacionalidad: diciembre y abril (Semana Santa) con mas viajes a top-5
    mask_dic = df['visit_date'].dt.month == 12
    mask_abr = df['visit_date'].dt.month == 4
    df.loc[mask_dic, 'destination'] = np.random.choice(destinations[:5], mask_dic.sum())
    df.loc[mask_abr, 'destination'] = np.random.choice(destinations[:5], mask_abr.sum())

    df.to_csv(save_path, index=False)
    print(f'Dataset sintetico guardado en {save_path}  ({len(df):,} filas)')
    return df

# Intentar kagglehub; ante cualquier fallo usar datos sinteticos
df = None
try:
    import kagglehub
    path = kagglehub.dataset_download('fajarpanji/tourism-travel-record')
    csv_files = list(Path(path).rglob('*.csv'))
    if csv_files:
        df = pd.read_csv(csv_files[0])
        df['visit_date'] = pd.to_datetime(df.get('visit_date', df.iloc[:, 0]))
        print(f'Dataset descargado desde Kaggle: {csv_files[0]}')
    else:
        raise FileNotFoundError('Sin CSVs en el dataset de Kaggle')
except Exception as e:
    print(f'kagglehub no disponible ({e}). Generando datos sinteticos...')
    df = generate_synthetic_data()

df['visit_date'] = pd.to_datetime(df['visit_date'])
print(f'\nDataset listo: {df.shape[0]:,} filas x {df.shape[1]} columnas')

## 1. Descripcion General del Dataset

In [ ]:
# ─── Celda 3: Forma, tipos de datos y primeras filas ─────────────────────────
print('='*55)
print(f'  Dimensiones : {df.shape[0]:,} filas  x  {df.shape[1]} columnas')
print('='*55)
print('\nTipos de datos:')
print(df.dtypes.to_string())
print('\nPrimeras 5 filas:')
df.head()

In [ ]:
# ─── Celda 4: Estadisticas descriptivas de variables numericas ───────────────
print('Estadisticas descriptivas (variables numericas):')
df.describe().round(2)

## 2. Distribucion de Viajes por Destino (Top 15)

In [ ]:
# ─── Celda 5: Barras – top 15 destinos mas visitados ─────────────────────────
top15 = df['destination'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(top15.index, top15.values,
              color=sns.color_palette('Blues_r', len(top15)))
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_title('Top 15 Destinos Turisticos por Numero de Viajes')
ax.set_xlabel('Destino')
ax.set_ylabel('Numero de viajes')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_top15_destinos.png')
plt.show()
print('Figura guardada: fig_top15_destinos.png')

## 3. Serie Temporal de Demanda Agregada

In [ ]:
# ─── Celda 6: Demanda diaria total (todos los destinos) ──────────────────────
# Se agrupa por fecha y se rellena dias sin registros con cero
demand_total = (
    df.groupby('visit_date')
      .size()
      .rename('total_viajes')
      .asfreq('D', fill_value=0)
      .reset_index()
)
demand_total['ma30'] = demand_total['total_viajes'].rolling(30).mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Panel superior: serie diaria cruda
axes[0].plot(demand_total['visit_date'], demand_total['total_viajes'],
             lw=0.8, color='steelblue', alpha=0.7)
axes[0].set_title('Demanda Diaria Total de Viajes')
axes[0].set_ylabel('Viajes/dia')

# Panel inferior: serie + media movil 30 dias
axes[1].plot(demand_total['visit_date'], demand_total['total_viajes'],
             lw=0.6, color='steelblue', alpha=0.4, label='Diario')
axes[1].plot(demand_total['visit_date'], demand_total['ma30'],
             lw=2.0, color='orangered', label='Media movil 30 d')
axes[1].set_title('Demanda con Media Movil 30 Dias')
axes[1].set_ylabel('Viajes/dia')
axes[1].set_xlabel('Fecha')
axes[1].legend()

plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_serie_temporal_total.png')
plt.show()
print('Figura guardada: fig_serie_temporal_total.png')

## 4. Analisis de Estacionalidad

In [ ]:
# ─── Celda 7: Estacionalidad mensual y por dia de semana ─────────────────────
# Se extrae mes y dia de semana para detectar patrones ciclicos
df['mes']        = df['visit_date'].dt.month
df['dia_semana'] = df['visit_date'].dt.dayofweek  # 0=lunes
df['anio']       = df['visit_date'].dt.year

meses_labels = ['Ene','Feb','Mar','Abr','May','Jun',
                'Jul','Ago','Sep','Oct','Nov','Dic']
dias_labels  = ['Lun','Mar','Mie','Jue','Vie','Sab','Dom']

viajes_mes = df.groupby('mes').size()
viajes_dia = df.groupby('dia_semana').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(meses_labels, viajes_mes.values,
            color=sns.color_palette('Oranges', 12))
axes[0].set_title('Distribucion Mensual de Viajes')
axes[0].set_xlabel('Mes')
axes[0].set_ylabel('Numero de viajes')

axes[1].bar(dias_labels, viajes_dia.values,
            color=sns.color_palette('Purples', 7))
axes[1].set_title('Distribucion por Dia de la Semana')
axes[1].set_xlabel('Dia')
axes[1].set_ylabel('Numero de viajes')

plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_estacionalidad.png')
plt.show()
print('Figura guardada: fig_estacionalidad.png')

In [ ]:
# ─── Celda 8: Heatmap mes x dia de semana ────────────────────────────────────
# Este heatmap permite identificar combinaciones de alta/baja demanda
pivot_heatmap = df.groupby(['mes', 'dia_semana']).size().unstack(fill_value=0)
pivot_heatmap.index   = meses_labels
pivot_heatmap.columns = dias_labels

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_heatmap, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, ax=ax)
ax.set_title('Heatmap de Viajes: Mes x Dia de la Semana')
ax.set_xlabel('Dia de la semana')
ax.set_ylabel('Mes')
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_heatmap_mes_dia.png')
plt.show()
print('Figura guardada: fig_heatmap_mes_dia.png')

## 5. Top 5 Rutas / Destinos Mas Populares

In [ ]:
# ─── Celda 9: Top 5 destinos con sus series temporales individuales ───────────
top5_names = df['destination'].value_counts().head(5).index.tolist()
print('Top 5 destinos:', top5_names)

fig, axes = plt.subplots(5, 1, figsize=(14, 18))
palette = sns.color_palette('tab10', 5)

for idx, (dest, ax) in enumerate(zip(top5_names, axes)):
    serie = (
        df[df['destination'] == dest]
          .groupby('visit_date').size()
          .rename('demand')
          .asfreq('D', fill_value=0)
    )
    ax.plot(serie.index, serie.values, lw=0.9, color=palette[idx], alpha=0.8)
    ma14 = serie.rolling(14).mean()
    ax.plot(serie.index, ma14.values, lw=2, color='black', alpha=0.6, label='MA-14')
    ax.set_title(f'Demanda diaria – {dest}')
    ax.set_ylabel('Viajes')
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_top5_series.png')
plt.show()
print('Figura guardada: fig_top5_series.png')

In [ ]:
# ─── Celda 10: Estadisticas detalladas del top 5 ─────────────────────────────
stats_top5 = (
    df[df['destination'].isin(top5_names)]
      .groupby('destination')
      .agg(
          total_viajes   = ('user_id',       'count'),
          rating_prom    = ('rating',        'mean'),
          costo_prom_usd = ('avg_cost_usd',  'mean'),
          duracion_prom  = ('duration_days', 'mean'),
          fecha_min      = ('visit_date',    'min'),
          fecha_max      = ('visit_date',    'max'),
      )
      .round(2)
      .sort_values('total_viajes', ascending=False)
)
print('Estadisticas del Top 5 destinos:')
stats_top5

## 6. Heatmap de Correlaciones

In [ ]:
# ─── Celda 11: Correlacion entre variables numericas ─────────────────────────
# Se usa triangulo inferior para evitar redundancia visual
num_cols = [c for c in df.select_dtypes(include='number').columns
            if c not in ('mes', 'dia_semana', 'anio')]
corr_matrix = df[num_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5,
            vmin=-1, vmax=1, ax=ax)
ax.set_title('Matriz de Correlacion – Variables Numericas')
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_correlaciones.png')
plt.show()
print('Figura guardada: fig_correlaciones.png')

## 7. Analisis de Valores Faltantes

In [ ]:
# ─── Celda 12: Valores faltantes por columna ─────────────────────────────────
missing = pd.DataFrame({
    'nulos':      df.isnull().sum(),
    'porcentaje': (df.isnull().mean() * 100).round(2)
})
missing_nonzero = missing[missing['nulos'] > 0].sort_values('porcentaje', ascending=False)

if missing_nonzero.empty:
    print('No hay valores faltantes en el dataset.')
else:
    print('Columnas con valores faltantes:')
    print(missing_nonzero)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(missing_nonzero.index, missing_nonzero['porcentaje'],
            color=sns.color_palette('Reds_r', len(missing_nonzero)))
    ax.set_xlabel('% Valores faltantes')
    ax.set_title('Valores Faltantes por Columna')
    plt.tight_layout()
    fig.savefig(BASE_DIR / 'fig_missing_values.png')
    plt.show()

print(f'\nTotal de valores faltantes en el dataset: {df.isnull().sum().sum()}')

In [ ]:
# ─── Celda 13: Distribucion de variables categoricas clave ───────────────────
# Graficos de torta para conocer la proporcion de cada categoria
cat_cols = [c for c in ['category', 'travel_type', 'season'] if c in df.columns]

if cat_cols:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 5))
    if len(cat_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        counts = df[col].value_counts()
        ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
               startangle=90, colors=sns.color_palette('pastel', len(counts)))
        ax.set_title(col.replace('_', ' ').title())
    plt.suptitle('Distribucion de Variables Categoricas', y=1.02, fontsize=14)
    plt.tight_layout()
    fig.savefig(BASE_DIR / 'fig_categoricas.png')
    plt.show()
    print('Figura guardada: fig_categoricas.png')

In [ ]:
# ─── Celda 14: Distribucion de variables numericas continuas ─────────────────
# Histogramas para identificar la forma de cada distribucion
cont_cols = [c for c in ['rating', 'avg_cost_usd', 'duration_days', 'num_reviews']
             if c in df.columns]

fig, axes = plt.subplots(1, len(cont_cols), figsize=(5 * len(cont_cols), 4))
for ax, col in zip(axes, cont_cols):
    ax.hist(df[col].dropna(), bins=30,
            color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
plt.suptitle('Distribuciones de Variables Numericas', y=1.02, fontsize=14)
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_distribuciones_num.png')
plt.show()
print('Figura guardada: fig_distribuciones_num.png')

In [ ]:
# ─── Celda 15: Evolucion anual de la demanda por destino (top 5) ──────────────
# Barras agrupadas por anio para cada uno de los 5 principales destinos
anual_top5 = (
    df[df['destination'].isin(top5_names)]
      .groupby(['anio', 'destination'])
      .size()
      .unstack(fill_value=0)
)

fig, ax = plt.subplots(figsize=(10, 5))
anual_top5.plot(kind='bar', ax=ax,
                color=sns.color_palette('tab10', 5),
                width=0.7, edgecolor='white')
ax.set_title('Evolucion Anual de Viajes – Top 5 Destinos')
ax.set_xlabel('Anio')
ax.set_ylabel('Numero de viajes')
ax.legend(title='Destino', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
fig.savefig(BASE_DIR / 'fig_evolucion_anual.png')
plt.show()
print('Figura guardada: fig_evolucion_anual.png')

In [ ]:
# ─── Celda 16: Box-plot de calificaciones por categoria ──────────────────────
# Permite comparar la satisfaccion del turista segun tipo de destino
if 'category' in df.columns and 'rating' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    order = df.groupby('category')['rating'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x='category', y='rating',
                order=order, palette='Set2', ax=ax)
    ax.set_title('Distribucion de Calificaciones por Categoria')
    ax.set_xlabel('Categoria')
    ax.set_ylabel('Rating')
    plt.tight_layout()
    fig.savefig(BASE_DIR / 'fig_rating_categoria.png')
    plt.show()
    print('Figura guardada: fig_rating_categoria.png')

In [ ]:
# ─── Celda 17: Resumen estadistico final ─────────────────────────────────────
print('=' * 60)
print('           RESUMEN ESTADISTICO DEL DATASET')
print('=' * 60)
print(f'  Total de registros          : {len(df):>8,}')
print(f'  Rango temporal              : {df["visit_date"].min().date()} -> {df["visit_date"].max().date()}')
print(f'  Numero de destinos unicos   : {df["destination"].nunique():>8}')
print(f'  Usuarios unicos             : {df["user_id"].nunique():>8}')
print(f'  Rating promedio             : {df["rating"].mean():>8.2f}')
if 'avg_cost_usd' in df.columns:
    print(f'  Costo promedio (USD)        : {df["avg_cost_usd"].mean():>8.1f}')
if 'duration_days' in df.columns:
    print(f'  Duracion promedio (dias)    : {df["duration_days"].mean():>8.1f}')
print(f'  Valores faltantes totales   : {df.isnull().sum().sum():>8,}')
print('=' * 60)
print('\nTop 5 destinos:')
for i, (dest, cnt) in enumerate(df['destination'].value_counts().head(5).items(), 1):
    print(f'  {i}. {dest:<20} {cnt:>6,} viajes')

## 8. Conclusiones del EDA

A partir del analisis exploratorio se destacan los siguientes hallazgos:

1. **Destinos populares**: Los cinco destinos con mayor demanda concentran una fraccion significativa del total de viajes, lo que justifica modelar individualmente cada ruta.

2. **Estacionalidad marcada**: Se observan picos claros en **diciembre** (vacaciones de fin de anio) y **abril** (Semana Santa), comportamiento tipico del turismo colombiano interno.

3. **Variacion semanal**: Los fines de semana (sabado y domingo) registran mayor actividad turistica, especialmente para destinos de playa y ecoturismo.

4. **Correlaciones bajas**: Las variables numericas (rating, costo, duracion) presentan correlaciones debiles entre si, lo que indica que la demanda no puede explicarse por una sola variable aislada.

5. **Datos completos**: El dataset no contiene valores faltantes; en datos reales seria necesaria una estrategia de imputacion (interpolacion lineal o LOCF).

6. **Senal para LSTM**: La serie temporal de demanda diaria muestra estructura temporal (tendencia + estacionalidad) susceptible de modelarse con redes LSTM con ventana deslizante de 30 dias.

### Proximo paso
Con estos insights, el **Notebook 02** construira y entrenara el modelo LSTM para predecir la demanda en los Top 5 destinos.

In [ ]:
# ─── Celda 18: Guardar dataset limpio para uso posterior ─────────────────────
clean_path = DATA_DIR / 'travel_data_clean.csv'
df.to_csv(clean_path, index=False)
print(f'Dataset limpio guardado en: {clean_path.resolve()}')
print('\nFiguras generadas en modulo1_demanda/:')
for f in sorted(Path('.').glob('fig_*.png')):
    print(f'  {f.name}')
print('\nEDA completado exitosamente.')